<a href="https://colab.research.google.com/github/gitmystuff/DSChunks/blob/main/Eigenvectors_and_Eigenvalues.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eigenvectors & Eigenvalues: The Math Behind PCA, Worked by Hand

* PCA finds the directions along which data varies the most. Those directions are the **eigenvectors** of the data's covariance matrix; how much variance lies along each direction is that eigenvector's **eigenvalue**.
* An eigenvector is a *direction*, expressed as one coefficient per original feature. An eigenvalue is a *single number* -- the variance of the data once projected onto that direction. They always come in matched pairs, one pair per feature.
* This notebook walks through the full mechanics on a tiny, 2-feature, 8-row dataset -- small enough to check every step with a calculator -- then verifies each hand-computed step against `numpy`.
* The defining equation for an eigenvector/eigenvalue pair of a matrix $\Sigma$ is: $$\Sigma \mathbf{v} = \lambda \mathbf{v}$$ -- "acting on $\mathbf{v}$ with $\Sigma$ just scales it by $\lambda$, without changing its direction."


## Step 1: The data

Two features -- `years_experience` and `skill_count` -- across 8 observations.


In [ ]:
import numpy as np

years_experience = np.array([2, 4, 5, 7, 8, 10, 12, 14])
skill_count       = np.array([3, 5, 4, 8, 7, 11, 10, 15])

X = np.column_stack([years_experience, skill_count])
print(X)


[[ 2  3]
 [ 4  5]
 [ 5  4]
 [ 7  8]
 [ 8  7]
 [10 11]
 [12 10]
 [14 15]]


## Step 2: Standardize

Subtract each column's mean, divide by its standard deviation, so both features are on the same scale before comparing their spread.


In [ ]:
X_mean = X.mean(axis=0)
X_std = X.std(axis=0, ddof=1)
X_scaled = (X - X_mean) / X_std
print(np.round(X_scaled, 3))


[[-1.403 -1.214]
 [-0.915 -0.716]
 [-0.671 -0.965]
 [-0.183  0.031]
 [ 0.061 -0.218]
 [ 0.549  0.778]
 [ 1.037  0.529]
 [ 1.525  1.774]]


## Step 3: Covariance matrix, by hand

Both standardized columns have variance exactly 1 (that fills the diagonal). For the covariance between them, multiply each row's pair together, sum, divide by $n-1=7$:

$$\text{cov} = \frac{\sum (x_i)(y_i)}{n-1} = \frac{6.669}{7} = 0.953$$

$$\Sigma = \begin{pmatrix} 1.000 & 0.953 \\ 0.953 & 1.000 \end{pmatrix}$$


In [ ]:
# Verify against numpy
cov_matrix = np.cov(X_scaled.T)
print(np.round(cov_matrix, 3))


[[1.    0.953]
 [0.953 1.   ]]


## Step 4: The eigenvalue equation

An eigenvector/eigenvalue pair must satisfy $\Sigma \mathbf{v} = \lambda \mathbf{v}$, which rearranges to $(\Sigma - \lambda I)\mathbf{v} = 0$. For a nonzero $\mathbf{v}$ to solve this, $(\Sigma - \lambda I)$ must be singular: $\det(\Sigma - \lambda I) = 0$.

First, build $\Sigma - \lambda I$ by subtracting $\lambda$ from just the diagonal (since $\lambda I$ is $\lambda$ on the diagonal, 0 elsewhere):

$$\Sigma - \lambda I = \begin{pmatrix} 1.000 & 0.953 \\ 0.953 & 1.000 \end{pmatrix} + \begin{pmatrix} -\lambda & 0 \\ 0 & -\lambda \end{pmatrix} = \begin{pmatrix} 1.000-\lambda & 0.953 \\ 0.953 & 1.000-\lambda \end{pmatrix}$$

Now set its determinant to zero and solve for $\lambda$:

$$(1-\lambda)^2 - (0.953)^2 = 0 \implies 1-\lambda = \pm 0.953$$

$$\lambda_1 = 1 - 0.953 = 0.047 \qquad \lambda_2 = 1 + 0.953 = 1.953$$

PC1 takes the **larger** eigenvalue: $\lambda = 1.953$.


**NOTE:** det, from $\det(\Sigma - \lambda I) = 0$, refers to the determinant.

**The determinant tells you how much a matrix stretches or squishes space — and whether it flattens it completely.**

Picture a square drawn on graph paper, with an area of 1. If you apply a matrix to every point in that square (a standard thing matrices do — transforming space), the square turns into some new shape — maybe a bigger square, a tilted parallelogram, whatever. The determinant is just the number that tells you **how much bigger or smaller that shape's area became** compared to the original.

- Determinant of 3 → the transformation triples the area
- Determinant of 1 → the area stays exactly the same (though the shape might rotate or tilt)
- Determinant of 0.5 → the area shrinks to half
- **Determinant of 0** → this is the important one — it means the transformation **squishes the shape flat**, collapsing it down to a line (or even a single point) with zero area. Nothing is left standing in more than one dimension.

That last case — determinant equals zero — is exactly why it matters for eigenvalues. When we solve $\det(\Sigma - \lambda I) = 0$, we're literally asking: "at what value of $\lambda$ does this matrix squash space flat instead of just stretching it?" Those special flattening points are exactly where a genuine eigenvector can exist — because eigenvectors are the one surviving **direction** that doesn't get flattened away, it just gets stretched (that stretch amount is the eigenvalue).

One thing to keep in mind: A **direction** is just a set of instructions, not a picture. In 3D, **direction** happens to look like an arrow, but really it's just three numbers: "move this much along x, this much along y, this much along z." Nothing about that idea requires stopping at 3(D) — it's just as valid to say "move this much along years_experience, this much along skill_count, this much along remote_ratio, ..." for 17 numbers instead of 3. You lose the picture, but the instructions still make complete sense.

In [ ]:
# Verify against numpy's eigenvalue solver
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
order = np.argsort(eigenvalues)[::-1]      # largest first
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

print("Eigenvalues (largest first):", np.round(eigenvalues, 3))


Eigenvalues (largest first): [1.953 0.047]


## Step 5: Solve for the matching eigenvector

Plug $\lambda = 1.953$ back into $(\Sigma - \lambda I)\mathbf{v} = 0$ and use the top row:

$$(1 - 1.953)v_1 + 0.953\,v_2 = 0 \implies -0.953\,v_1 + 0.953\,v_2 = 0 \implies v_1 = v_2$$

Any vector with equal entries works; normalize to length 1 by dividing by $\sqrt{2}$:

$$\mathbf{v}_1 = (0.707,\ 0.707)$$

This makes sense: the two features are so strongly correlated (covariance 0.953) that PC1 weighs them equally.


In [ ]:
pc1_vector = eigenvectors[:, 0]
if pc1_vector[0] < 0:          # sign is arbitrary -- flip for a clean positive comparison
    pc1_vector = -pc1_vector
print("PC1 eigenvector:", np.round(pc1_vector, 3))


PC1 eigenvector: [0.707 0.707]


## Step 6: Compute PC1 scores by hand

Each row's PC1 score is the dot product of its standardized values with the eigenvector.

Row 0: $(-1.403)(0.707) + (-1.214)(0.707) = -0.992 - 0.858 = -1.850$

Row 7: $(1.525)(0.707) + (1.774)(0.707) = 1.078 + 1.254 = 2.332$


In [ ]:
pc1_scores = X_scaled @ pc1_vector
print("PC1 score per row:", np.round(pc1_scores, 3))


PC1 score per row: [-1.851 -1.153 -1.157 -0.107 -0.111  0.939  1.108  2.333]


## Step 7: Verify

**Check 1 -- the eigenvalue equals the variance of the projected scores:**


In [ ]:
print(f"Variance of PC1 scores: {np.var(pc1_scores, ddof=1):.3f}")
print(f"PC1 eigenvalue:        {eigenvalues[0]:.3f}")


Variance of PC1 scores: 1.953
PC1 eigenvalue:        1.953


**Check 2 -- the defining equation itself, $\Sigma \mathbf{v} = \lambda \mathbf{v}$, holds:**

In [ ]:
left_side = cov_matrix @ pc1_vector
right_side = eigenvalues[0] * pc1_vector

print("Sigma . v  =", np.round(left_side, 3))
print("lambda . v =", np.round(right_side, 3))


Sigma . v  = [1.381 1.381]
lambda . v = [1.381 1.381]


Both checks match, confirming everything computed by hand above -- the covariance matrix, the eigenvalue, and the eigenvector -- is internally consistent. This is the exact mechanism `sklearn.decomposition.PCA` runs (on however many features and rows you actually have) every time you call `.fit_transform()`.
